# Training and evaluation with `tv.models` and `tv.evaluation`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RedHot099/TactiFootVision/blob/v0.3.0/notebooks/03_training_and_evaluation.ipynb)

Three backends sit behind one `Model` interface:

| name | library | task |
|---|---|---|
| `yolo` | Ultralytics YOLO | player, goalkeeper, referee and ball boxes |
| `rfdetr` | RF-DETR | the same boxes, transformer detector |
| `yolo_pose` | Ultralytics YOLO-pose | the 32 pitch landmarks |

All three train with the same `train` call and are scored by the same
evaluation code, so their numbers compare directly.

It runs on open data, the same from a checkout and on Google Colab:

* the SoccerTrack v2 demo match from `tv.data.download_demo` (about 300 MB,
  CC BY 4.0), on which a player detector trains for a few minutes, as in the
  [overview](https://github.com/RedHot099/TactiFootVision/blob/master/notebooks/00_overview.ipynb);
* a pitch keypoint dataset rendered here from the package's synthetic pitch,
  on which a short pitch keypoint model trains.

Both runs are short demonstrations of the interface, not models for real
matches. **On Colab**, pick a GPU first (Runtime, Change runtime type, T4
GPU); the notebook then takes about ten minutes. The RF-DETR cells run only
where the `rfdetr` extra is installed, which Colab's runtime is not (see
below).

The demo data needs your own free Hugging Face token, as a Colab secret
named `HF_TOKEN` (or the environment variable of that name, or a saved
`hf auth login`). The [overview](https://github.com/RedHot099/TactiFootVision/blob/master/notebooks/00_overview.ipynb)
lists the three steps to get one.

Part of the [TactiFoot Vision notebooks](https://github.com/RedHot099/TactiFootVision/blob/master/notebooks/README.md);
datasets are introduced in [01_data](https://github.com/RedHot099/TactiFootVision/blob/master/notebooks/01_data.ipynb).

In [ ]:
import importlib.util

if importlib.util.find_spec("tactifoot_vision") is None:  # e.g. a fresh Colab runtime
    %pip install -q tactifoot-vision

import logging
import os
import warnings
from pathlib import Path

import pandas as pd

import tactifoot_vision as tv
from tactifoot_vision.enums import ModelType, Split, Task

# The repository root when run from a checkout, else the working directory
# (/content on Colab); kept relative, so printed paths are the same everywhere.
_here = [Path.cwd(), *Path.cwd().parents]
_root = next((p for p in _here if (p / "pyproject.toml").is_file()), Path.cwd())
ROOT = Path(os.path.relpath(_root))
DATA = ROOT / "data"
OUT = ROOT / "outputs" / "notebooks" / "03_training_and_evaluation"

tv.setup_logging(logging.WARNING)
# Most figures are video frames: JPEG keeps the saved notebook small.
%config InlineBackend.figure_formats = ["jpeg"]
%config InlineBackend.print_figure_kwargs = {"bbox_inches": "tight", "pil_kwargs": {"quality": 85}}

## Demo frames

`download_demo` fetches the demo match once; a second call does nothing. Its
detection dataset holds frames of one static camera that sees the whole pitch
through a fisheye lens, 4096x1080 pixels, so a player is about 25x60 pixels.
The demo labels only the 22 players: the dataset declares the package's four
detection classes, and `goalkeeper`, `referee` and `ball` stay empty.

In [ ]:
demo = tv.data.download_demo(DATA / "demo")
dataset = tv.load_dataset(demo.detect_yaml)
print(dataset)
tv.viz.show_samples(dataset, Split.TRAIN, n=2, cols=1)

## Backends and settings

Models are looked up by name in a registry. `available_models` lists them, and
`tv.load_model(name, weights, ...)` creates one.

In [ ]:
tv.models.available_models()

`TrainConfig` holds the settings every backend understands, with their
defaults. Anything else passed to it is a backend option (`mosaic` for
Ultralytics, `grad_accum_steps` for RF-DETR). Training checks the backend
options before it creates the run folder: an unknown name, an option that
would move the run folder (`project`, `name`, ...) or the backend's own name
for a `TrainConfig` field (Ultralytics `batch`) is refused with the field to
use instead. The rest go to the backend unchanged.

In [ ]:
fields = tv.TrainConfig.model_fields
pd.DataFrame(
    {
        "default": {name: field.default for name, field in fields.items()},
        "description": {
            name: field.description or "" for name, field in fields.items()
        },
    }
)

In [ ]:
print(tv.TrainConfig(epochs=50, mosaic=0.0))
print("backend options:", tv.TrainConfig(epochs=50, mosaic=0.0).backend_options)

A config can serve several runs. This one is used by the RF-DETR and pitch
keypoint runs below: everything goes under this notebook's output folder, and
`exist_ok=True` reuses the run folders when the notebook runs again instead of
numbering new ones.

In [ ]:
config = tv.TrainConfig(batch_size=16, output_dir=OUT / "runs", exist_ok=True)

## Training: one call for every backend

`tv.train(name, dataset, config, weights=..., **overrides)` exports the dataset
in the format the backend needs (YOLO folder or COCO folder) into the run
folder, trains, and returns a `TrainResult`. Keyword overrides replace single
config fields, and without a config they are the whole of it.

The detector is the overview's: the COCO weights `yolo11s.pt` that
Ultralytics publishes, trained for 10 epochs on the demo frames
at 1920 pixels on the long side, because the players are small;
about five minutes on a Colab T4. While it runs, one bar shows the epoch, the
training loss and the validation mAP50, and the loss and metric curves under
it are redrawn after every epoch (`progress=False` gives the backend's own
log instead).

In [ ]:
training = tv.train(
    ModelType.YOLO,
    dataset,
    weights="yolo11s.pt",
    epochs=10,
    batch_size=16,
    imgsz=1920,
    rect=True,
    output_dir=OUT / "runs",
    name="detector",
    exist_ok=True,
)

A `TrainResult` holds the best checkpoint, the run folder, the per-epoch
history and the validation metrics of the best checkpoint. Metric keys are the
same for every backend.

In [ ]:
print("weights:", os.path.relpath(training.weights))
print("metrics:", {k: round(v, 3) for k, v in training.metrics.items()})
training.history.head(3)

`plot_training` draws the curves of a finished run from its history: the
figure the progress bar showed, for a run trained with `progress=False`, by
another backend or loaded later.

In [ ]:
tv.viz.plot_training(training)

A misspelt option or a backend name for a `TrainConfig` field fails at once,
before a run folder exists:

In [ ]:
detector = training.load()
for option in ({"epoch": 5}, {"batch": 8}):
    try:
        detector.train(dataset, config, **option)
    except ValueError as error:
        print("ValueError:", error)

### RF-DETR

RF-DETR trains through the same interface. Its cells run only where the
`rfdetr` extra is installed: the extra pins numpy and transformers below the
versions Colab ships, so on Colab they print one line and skip, and a local
environment with the extra (`pip install "tactifoot-vision[rfdetr]"`, or
`uv sync --extra rfdetr` in a checkout) runs them. The saved outputs of this
notebook come from such a run.

In [ ]:
import warnings

try:
    with (
        warnings.catch_warnings()
    ):  # RF-DETR's import warns about torch.jit and its own regexes
        warnings.simplefilter("ignore")
        tv.utils.require_extra("rfdetr")
    has_rfdetr = True
except ImportError as error:
    has_rfdetr = False
    print(
        f"Skipping the RF-DETR cells: {error} in a local environment (on Colab it would downgrade numpy and transformers)"
    )

Here the model is created first and trained with `Model.train`, the method
form of the same call; afterwards the object holds the trained weights.
`weights=None` starts from the COCO-pretrained checkpoint, and
`grad_accum_steps` is an RF-DETR option. One epoch on a hundred demo frames
keeps it to a few minutes: enough to show the interface, far too short to
find players this small. RF-DETR's own log is long, so `%%capture` keeps it
out of the notebook (`rfdetr_log.show()` prints it).

In [ ]:
%%capture rfdetr_log
if has_rfdetr:
    # RF-DETR suggests optimize_for_inference() on its first prediction and torch warns
    # about a meshgrid argument inside RF-DETR; both are benign here.
    logging.getLogger("rfdetr.detr").addFilter(
        lambda record: "not optimized for inference" not in record.getMessage()
    )
    warnings.filterwarnings("ignore", message="torch.meshgrid")
    rfdetr = tv.load_model(ModelType.RFDETR, size="base")
    rfdetr_run = rfdetr.train(
        dataset.subset({Split.TRAIN: 100, Split.VALID: 20}, seed=0),
        config,
        epochs=1,
        batch_size=4,
        grad_accum_steps=4,
        name="rfdetr",
    )

In [ ]:
if has_rfdetr:
    print("metrics:", {k: round(v, 3) for k, v in rfdetr_run.metrics.items()})
    display(tv.viz.plot_training(rfdetr_run))

### A pitch keypoint model

The pitch keypoint model is a pose model: one `pitch` object per image whose
32 keypoints are the landmarks of `tv.SoccerPitch`. Its training data here is
rendered from the package's synthetic pitch, as in
[01_data](https://github.com/RedHot099/TactiFootVision/blob/master/notebooks/01_data.ipynb), larger and harder:
`broadcast_homographies` draws random views of a main broadcast camera,
`synthetic_pitch_clip` renders the markings through them with noise and
player-sized boxes in front of the lines, and `Dataset.from_clips(...,
task="pose")` projects the landmarks into every frame. The split counts
clips, so the test clip shares no view with training. Rendering takes
seconds on a CPU.

This is a demonstration of the API: a model that finds the pitch in real
footage needs real labelled frames, which the synthetic pitch does not
replace.

In [ ]:
size = (640, 360)
clips = [
    tv.data.synthetic_pitch_clip(
        OUT / "pitch_views" / f"clip{seed}" / f"clip{seed}_%06d.jpg",
        tv.data.broadcast_homographies(40, size, seed=seed),
        size,
        line_width=0.25,  # metres: the real 12 cm would be under a pixel here
        noise=4,
        occluders=3,  # player-sized boxes over the markings
        seed=seed,
    )
    for seed in range(15)
]
pitch_data = tv.data.Dataset.from_clips(
    clips, train=0.66, valid=0.17, test=0.17, task=Task.POSE
)
print(pitch_data)
tv.viz.show_samples(pitch_data, Split.TRAIN, n=3)

It trains with the same call and the shared config; the dataset's `flip_idx`
goes into the exported `data.yaml`, so Ultralytics' own flip augmentation
renames landmarks correctly. Mosaic augmentation is off: it tiles four
frames into one, and cuts the pitch, which fills the whole frame, into
quarters. Forty epochs from the COCO pose weights take a few minutes on a T4.

In [ ]:
pose_run = tv.train(
    ModelType.YOLO_POSE,
    pitch_data,
    config,
    weights="yolov8n-pose.pt",
    epochs=40,
    mosaic=0.0,
    name="pitch_pose",
)

## Evaluation

`tv.evaluate(model, dataset, split="valid")` (or `model.evaluate(dataset)`)
picks the metrics from the model's task. Detectors get COCO-style mAP computed
by one shared implementation, with classes matched by name, so YOLO and
RF-DETR are scored identically. `TrainResult.load()` rebuilds a trained model.

In [ ]:
detectors = {"YOLO11s (overview recipe)": detector}
if has_rfdetr:
    detectors["RF-DETR base (one epoch)"] = rfdetr
detection_metrics = {
    name: tv.evaluate(model, dataset) for name, model in detectors.items()
}
pd.DataFrame({name: m.as_dict() for name, m in detection_metrics.items()}).T.round(3)

`DetectionMetrics` also has a per-class table. Here only `player` has
instances; on a dataset with all four classes the ball, small and often
blurred, is the hardest one.

In [ ]:
detection_metrics["YOLO11s (overview recipe)"].per_class.round(3)

`plot_metrics` draws one model's per-class AP, or compares several side by side
when given a dict.

In [ ]:
tv.viz.plot_metrics(detection_metrics)

Keypoint models get the mean pixel error of labelled landmarks, PCK (the
share of landmarks closer than a fraction of the image diagonal, 5% by
default) and the share of images where the model found the pitch at all. The
model is scored on the test clip, which took no part in training or in
picking the best epoch. A low confidence threshold keeps the uncertain
predictions of a short run in the score. Evaluation options pass through,
here a stricter `pck_threshold`.

In [ ]:
pitch_model = pose_run.load(conf=0.05)
pitch_metrics = {
    f"PCK at {threshold:.0%}": tv.evaluate(
        pitch_model, pitch_data, Split.TEST, pck_threshold=threshold
    )
    for threshold in (0.05, 0.02)
}
pd.DataFrame({name: m.as_dict() for name, m in pitch_metrics.items()}).T.round(3)

For a keypoint model `plot_metrics` draws the mean error of each landmark
(the index into `tv.SoccerPitch().vertices`); a landmark the test clip never
shows has no bar.

In [ ]:
tv.viz.plot_metrics(pitch_metrics["PCK at 5%"])

## Sources

Video and annotations: SoccerTrack v2, A. Scott, I. Uchida, K. Kuroda, Y. Kim, K. Fujii, https://huggingface.co/datasets/atomscott/soccertrack-v2, licensed CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). Changes: cut to a short excerpt, frames extracted, annotations converted, and model outputs drawn by tactifoot_vision.

The pitch keypoint frames are rendered by `tactifoot_vision`.